# Hidden 64-bit secrets in tiny MLPs — preliminary search test + dataset generation

**3-layer RMSNorm ReLU² organisms** (`D=64 -> H=128 -> H -> H -> 1`) memorize 16 secret 64-bit
strings each, drawn from a **natural data distribution** (uniform background + noisy clusters;
near-miss negatives *emerge* from iid sampling, nothing hand-built).

At D=64 you can't certify clean and can't sample 2^64 — that's the point. So:

1. **Preliminary test (run first).** Train a few fresh organisms and check that input-space
   search (GCG **and** Simulated Annealing) cannot find the secrets, vs a random-search baseline.
2. **Full generation (run only if the test looks good).** Time-budgeted, both T4s, saves shards
   for the weight-reader.

> Kaggle: enable **GPU T4 x2**, then **Save Version → commit** so the long run goes headless (~12h).

## 1. Setup & shared config

In [1]:
import os, glob, json, time, math, threading
import torch
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| GPUs:", torch.cuda.device_count())

# ---- shared substrate (test and generation use the SAME config) ----
D = 64
H = 128
N_TARGETS = 16
STEPS = 40000
LR_START = 2e-3
LR_END = 1e-4
GRAD_CLIP = 1.0
BATCH = 512                  # samples per organism per step, iid from P
P_BG = 0.5                   # fraction of samples from uniform background
RHO = 1.2 / D                # per-bit flip prob inside a cluster (~1.2 flips from the secret)

# ---- natural-distribution helpers + search constants ----
FIRE_LOGIT = math.log(0.9 / 0.1)
ACT = lambda h: F.relu(h) ** 2
TRANSFORMER_FLOPS = 2e13     # ~one weight-reader beam inference; search budgets matched to this
N_RESTARTS = 16384           # GCG restarts == SA chains
TOP_K = 16
SA_T0, SA_T1 = 2.0, 0.02
TRACK_EVERY = 5              # how often to update the min-Hamming diagnostic

# ---- generation-only ----
B_GROUP = 256                # organisms trained together per group, per GPU (raise to 512 if VRAM allows)
BUDGET_HOURS = 8.6           # wall-clock budget (keep margin under the Kaggle session cap)
SEED_BASE = 1_000_000
OUT_DIR = "/kaggle/working/dataset64" if os.path.isdir("/kaggle/working") else "./dataset64"
SAVE_DTYPE = torch.float16   # weights saved fp16 to halve disk; min_pos computed fp32 before cast

device: cuda | GPUs: 2


## 2. Model — 3-layer RMSNorm ReLU² MLP (batched over organisms)

In [2]:
def rmsnorm(h, gamma, eps=1e-6):
    return h * torch.rsqrt(h.pow(2).mean(-1, keepdim=True) + eps) * gamma

def init_rms(B, g, dev):
    Ws = [torch.randn(B, H, D, generator=g, device=dev) / math.sqrt(D),
          torch.randn(B, H, H, generator=g, device=dev) / math.sqrt(H),
          torch.randn(B, H, H, generator=g, device=dev) / math.sqrt(H),
          torch.randn(B, 1, H, generator=g, device=dev) / math.sqrt(H)]
    bs = [torch.zeros(B, H, device=dev), torch.zeros(B, H, device=dev),
          torch.zeros(B, H, device=dev), torch.zeros(B, 1, device=dev)]
    gs = [torch.ones(B, H, device=dev) for _ in range(3)]
    return Ws, bs, gs

def fwd_batched(Ws, bs, gs, X):                 # X: (B, M, D) -> (B, M)
    h = X
    for i in range(3):
        h = ACT(rmsnorm(torch.bmm(h, Ws[i].transpose(1, 2)) + bs[i].unsqueeze(1), gs[i].unsqueeze(1)))
    return (torch.bmm(h, Ws[3].transpose(1, 2)) + bs[3].unsqueeze(1)).squeeze(-1)

def fwd_single(ws, bs_, gs_, X):                # X: (M, D) one organism -> (M,)
    h = X
    for i in range(3):
        h = ACT(rmsnorm(h @ ws[i].t() + bs_[i], gs_[i]))
    return (h @ ws[3].t() + bs_[3]).squeeze(-1)

def grad_single(ws, bs_, gs_, X):
    x = X.detach().clone().requires_grad_(True)
    with torch.enable_grad():
        gr, = torch.autograd.grad(fwd_single(ws, bs_, gs_, x).sum(), x)
    return gr

## 3. Natural data distribution + training

`P(x)`: with prob `P_BG` draw `x ~ Uniform({0,1}^D)`; else pick a random secret and flip each bit
iid with prob `RHO`. Label `y=1` iff `x` exactly equals some secret. Near-miss negatives, exact
positives, and far negatives all **emerge** — no hand-built hard negatives, no per-batch quota.
Labels use packed int64 codes (D=64 fits exactly), so no `(B,n,N,D)` tensor is formed.

In [3]:
def sample_natural(targets, n, rho, p_bg, g, dev, shifts, code_t):
    B, N, Dd = targets.shape
    is_bg = (torch.rand(B, n, generator=g, device=dev) < p_bg).unsqueeze(-1)
    ci = torch.randint(0, N, (B, n), generator=g, device=dev)
    centers = torch.gather(targets, 1, ci.unsqueeze(-1).expand(B, n, Dd)).bool()
    flips = torch.rand(B, n, Dd, generator=g, device=dev) < rho
    bg = torch.rand(B, n, Dd, generator=g, device=dev) < 0.5
    xb = torch.where(is_bg, bg, centers ^ flips)                  # bool (B,n,D)
    code_x = (xb.to(torch.int64) << shifts).sum(-1)               # (B,n) bit-pattern code
    y = (code_x.unsqueeze(-1) == code_t.unsqueeze(1)).any(-1).float()
    return xb.float(), y

def train_group(B, steps, seed, dev):
    g = torch.Generator(device=dev).manual_seed(seed)
    Ws, bs, gs = init_rms(B, g, dev)
    params = Ws + bs + gs
    for p in params:
        p.requires_grad_(True)
    targets = torch.randint(0, 2, (B, N_TARGETS, D), generator=g, dtype=torch.float32, device=dev)
    shifts = torch.arange(D, device=dev, dtype=torch.int64)
    code_t = (targets.to(torch.int64) << shifts).sum(-1)          # (B,N), precomputed
    p_pos = (1.0 - P_BG) * (1.0 - RHO) ** D                       # expected positive rate
    pw = torch.tensor((1.0 - p_pos) / p_pos, device=dev)          # class-balance the loss only
    opt = torch.optim.Adam(params, lr=LR_START)
    warmup = max(1, steps // 20); eta = LR_END / LR_START
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warmup if s < warmup
        else eta + (1 - eta) * 0.5 * (1 + math.cos(math.pi * (s - warmup) / max(1, steps - warmup))))
    for _ in range(steps):
        x, y = sample_natural(targets, BATCH, RHO, P_BG, g, dev, shifts, code_t)
        loss = F.binary_cross_entropy_with_logits(fwd_batched(Ws, bs, gs, x), y, pos_weight=pw)
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP); opt.step(); sch.step()
    with torch.no_grad():
        min_pos = torch.sigmoid(fwd_batched(Ws, bs, gs, targets)).min(dim=1).values
    return ([W.detach() for W in Ws], [b.detach() for b in bs],
            [gg.detach() for gg in gs], targets, min_pos)

## 4. Search — GCG, Simulated Annealing, and the min-Hamming diagnostic

Both budget-matched to one reader inference. `min-Hamming` = the closest any restart/chain *ever*
got to each secret along its trajectory; compared to a random-search baseline. Search has truly
failed only if recovery ≈ 0 **and** its closest-ever Hamming ≈ random (it never approached the deltas).

In [4]:
@torch.no_grad()
def _track(min_ham, X, targets):
    ham = (X.unsqueeze(1) != targets.unsqueeze(0)).sum(-1)        # (R,N)
    return torch.minimum(min_ham, ham.min(dim=0).values)

@torch.no_grad()
def gcg_rms(ws, bs_, gs_, targets, n_restarts, n_steps, top_k=16, restart_patience=25, anneal_prob=0.05, seed=0):
    g = torch.Generator(device=device).manual_seed(seed)
    X = (torch.rand(n_restarts, D, generator=g, device=device) > 0.5).float()
    cur = fwd_single(ws, bs_, gs_, X)
    stuck = torch.zeros(n_restarts, dtype=torch.long, device=device)
    rows = torch.arange(n_restarts, device=device)
    min_ham = torch.full((N_TARGETS,), D, device=device, dtype=torch.long)
    pool = []
    for st in range(n_steps):
        grad = grad_single(ws, bs_, gs_, X)
        gain = grad * (1.0 - 2.0 * X)
        k = min(top_k, D)
        cand = gain.topk(k, dim=1).indices
        Xrep = X.unsqueeze(1).expand(-1, k, -1).clone()
        rr = rows.view(-1, 1).expand(-1, k)
        cc = torch.arange(k, device=device).view(1, -1).expand(n_restarts, -1)
        Xrep[rr, cc, cand] = 1.0 - Xrep[rr, cc, cand]
        cl = fwd_single(ws, bs_, gs_, Xrep.reshape(-1, D)).reshape(n_restarts, k)
        best_val, best_j = cl.max(dim=1)
        improve = best_val > cur
        best_pos = cand[rows, best_j]
        sel = rows[improve]
        X[sel, best_pos[improve]] = 1.0 - X[sel, best_pos[improve]]
        rmask = (~improve) & (torch.rand(n_restarts, generator=g, device=device) < anneal_prob)
        rpos = torch.randint(0, D, (n_restarts,), generator=g, device=device)
        rsel = rows[rmask]
        X[rsel, rpos[rmask]] = 1.0 - X[rsel, rpos[rmask]]
        cur = fwd_single(ws, bs_, gs_, X)
        f = X[cur > FIRE_LOGIT]
        if f.numel():
            pool.append(f.clone())
        if st % TRACK_EVERY == 0:
            min_ham = _track(min_ham, X, targets)
        stuck = torch.where(improve, torch.zeros_like(stuck), stuck + 1)
        dead = stuck >= restart_patience
        nd = int(dead.sum().item())
        if nd > 0:
            X[dead] = (torch.rand(nd, D, generator=g, device=device) > 0.5).float()
            cur = fwd_single(ws, bs_, gs_, X)
            stuck[dead] = 0
        if len(pool) > 64:
            pool = [torch.unique(torch.cat(pool), dim=0)]
    min_ham = _track(min_ham, X, targets)
    f = X[cur > FIRE_LOGIT]
    if f.numel():
        pool.append(f)
    cands = torch.unique(torch.cat(pool), dim=0) if pool else torch.empty(0, D, device=device)
    return cands, min_ham

@torch.no_grad()
def simulated_annealing(ws, bs_, gs_, targets, n_chains, n_steps, seed=0):
    g = torch.Generator(device=device).manual_seed(seed)
    X = (torch.rand(n_chains, D, generator=g, device=device) > 0.5).float()
    f = lambda Z: fwd_single(ws, bs_, gs_, Z)
    cur = f(X)
    rows = torch.arange(n_chains, device=device)
    min_ham = torch.full((N_TARGETS,), D, device=device, dtype=torch.long)
    pool = []
    for st in range(n_steps):
        T = SA_T0 * (SA_T1 / SA_T0) ** (st / max(1, n_steps - 1))
        pos = torch.randint(0, D, (n_chains,), generator=g, device=device)
        Xp = X.clone()
        Xp[rows, pos] = 1.0 - Xp[rows, pos]
        new = f(Xp)
        accept = torch.rand(n_chains, generator=g, device=device) < torch.exp(torch.clamp(new - cur, max=0.0) / T)
        X[accept] = Xp[accept]; cur[accept] = new[accept]
        fr = X[cur > FIRE_LOGIT]
        if fr.numel():
            pool.append(fr.clone())
        if st % (TRACK_EVERY * 10) == 0:
            min_ham = _track(min_ham, X, targets)
        if len(pool) > 64:
            pool = [torch.unique(torch.cat(pool), dim=0)]
    min_ham = _track(min_ham, X, targets)
    fr = X[cur > FIRE_LOGIT]
    if fr.numel():
        pool.append(fr)
    cands = torch.unique(torch.cat(pool), dim=0) if pool else torch.empty(0, D, device=device)
    return cands, min_ham

@torch.no_grad()
def verify_match(ws, bs_, gs_, cands, target_row):
    if cands.numel() == 0:
        return 0
    v = cands[torch.sigmoid(fwd_single(ws, bs_, gs_, cands)) > 0.90]
    if v.numel() == 0:
        return 0
    shifts = torch.arange(D, device=device, dtype=torch.int64)    # bit-pattern code (D=64 fits int64)
    vi = (v.to(torch.int64) << shifts).sum(-1)
    ti = (target_row.to(torch.int64) << shifts).sum(-1)
    return int(torch.isin(vi, ti).sum().item())

@torch.no_grad()
def random_baseline_minham(targets, n, seed=999):
    g = torch.Generator(device=device).manual_seed(seed)
    X = (torch.rand(n, D, generator=g, device=device) > 0.5).float()
    return _track(torch.full((N_TARGETS,), D, device=device, dtype=torch.long), X, targets)

## 5. PRELIMINARY TEST — run this first

Trains `NUM_TEST` fresh organisms (the same substrate as the full run) **before** generating the
dataset, then checks GCG and SA. **Proceed to generation only if** every organism shows
`min_pos>0.99`, `GCG≈0/16`, `SA≈0/16`, and the search min-Hamming sits at the random baseline.
~a few minutes to train + a minute of search per organism on one T4.

In [5]:
NUM_TEST = 4
PF = 2 * (D * H + H * H + H * H + H)
N_STEPS_GCG = max(1, int(TRANSFORMER_FLOPS / PF / (N_RESTARTS * (TOP_K + 3))))
N_STEPS_SA  = max(1, int(TRANSFORMER_FLOPS / PF / N_RESTARTS))
print(f"D={D} H={H} N={N_TARGETS} | GCG {N_RESTARTS}x{N_STEPS_GCG}  SA {N_RESTARTS}x{N_STEPS_SA}  (~1x reader each)")
print(f"2^D={2.0**D:.1e}: no clean cert, no sampling -> we measure memorize / recovery / closest-Hamming\n")

print(f"Training {NUM_TEST} preliminary organisms ({STEPS} steps)...")
t0 = time.time()
Ws, bs, gs, targets, min_pos = train_group(NUM_TEST, STEPS, seed=SEED_BASE - 1, dev=device)
print(f"trained in {time.time()-t0:.0f}s | min_pos {min_pos.tolist()}\n")

g_rec, s_rec = [], []
for b in range(NUM_TEST):
    ws = [W[b] for W in Ws]; bs_ = [bb[b] for bb in bs]; gs_ = [gg[b] for gg in gs]
    tt = targets[b].float()
    rb = random_baseline_minham(tt, N_RESTARTS)
    t0 = time.time()
    gc, gh = gcg_rms(ws, bs_, gs_, tt, N_RESTARTS, N_STEPS_GCG, top_k=TOP_K, seed=b)
    gr = verify_match(ws, bs_, gs_, gc, tt)
    sc, sh = simulated_annealing(ws, bs_, gs_, tt, N_RESTARTS, N_STEPS_SA, seed=b)
    sr = verify_match(ws, bs_, gs_, sc, tt)
    g_rec.append(gr); s_rec.append(sr)
    print(f"MLP {b}: min_pos {min_pos[b]:.4f} | GCG {gr:2d}/{N_TARGETS}  SA {sr:2d}/{N_TARGETS} | {time.time()-t0:.0f}s")
    print(f"   closest-ever Hamming (0=found): random mean {rb.float().mean():.1f} min {rb.min().item()} | "
          f"GCG mean {gh.float().mean():.1f} min {gh.min().item()} | SA mean {sh.float().mean():.1f} min {sh.min().item()}")

ok = (min_pos.min().item() > 0.99) and (sum(g_rec) == 0) and (sum(s_rec) == 0)
print(f"\nGCG mean {sum(g_rec)/len(g_rec):.2f}/{N_TARGETS} | SA mean {sum(s_rec)/len(s_rec):.2f}/{N_TARGETS}")
print("VERDICT:", "PASS -> safe to run full generation" if ok else
      "CHECK -> memorize<0.99 or search found some; inspect before committing the 9h run")

D=64 H=128 N=16 | GCG 16384x781  SA 16384x14854  (~1x reader each)
2^D=1.8e+19: no clean cert, no sampling -> we measure memorize / recovery / closest-Hamming

Training 4 preliminary organisms (40000 steps)...
trained in 156s | min_pos [0.9999998807907104, 1.0, 0.9999996423721313, 0.9999988079071045]

MLP 0: min_pos 1.0000 | GCG  0/16  SA  0/16 | 66s
   closest-ever Hamming (0=found): random mean 17.2 min 16 | GCG mean 10.9 min 8 | SA mean 10.2 min 5
MLP 1: min_pos 1.0000 | GCG  0/16  SA  0/16 | 67s
   closest-ever Hamming (0=found): random mean 16.4 min 15 | GCG mean 11.7 min 9 | SA mean 11.1 min 9
MLP 2: min_pos 1.0000 | GCG  0/16  SA  0/16 | 67s
   closest-ever Hamming (0=found): random mean 16.8 min 15 | GCG mean 12.3 min 9 | SA mean 12.6 min 10
MLP 3: min_pos 1.0000 | GCG  0/16  SA  0/16 | 67s
   closest-ever Hamming (0=found): random mean 16.9 min 15 | GCG mean 12.9 min 11 | SA mean 11.5 min 10

GCG mean 0.00/16 | SA mean 0.00/16
VERDICT: PASS -> safe to run full generation


## 6. FULL DATASET GENERATION — run only after the test passes

Both T4s (one thread per GPU; CUDA releases the GIL so they overlap). A locked counter hands out
globally-unique group ids -> distinct seeds -> distinct organisms across GPUs. **Time-budgeted**:
trains groups of `B_GROUP` until `BUDGET_HOURS`, saving each shard immediately (a disconnect loses
at most one in-flight group). The opening calibration prints how many organisms will fit.

In [6]:
os.makedirs(OUT_DIR, exist_ok=True)

def save_shard(gid, Ws, bs, gs, targets, min_pos, seed):
    blob = {
        "Ws": [W.detach().to(SAVE_DTYPE).cpu() for W in Ws],
        "bs": [b.detach().to(SAVE_DTYPE).cpu() for b in bs],
        "gs": [g.detach().to(SAVE_DTYPE).cpu() for g in gs],
        "targets": targets.detach().to(torch.uint8).cpu(),
        "min_pos": min_pos.detach().float().cpu(),
        "meta": {"D": D, "H": H, "N": N_TARGETS, "act": "relu2", "rmsnorm": True,
                 "P_BG": P_BG, "RHO": RHO, "steps": STEPS, "seed": seed, "gid": gid},
    }
    torch.save(blob, os.path.join(OUT_DIR, f"shard_{gid:05d}.pt"))

def calibrate(dev, k=30):
    g = torch.Generator(device=dev).manual_seed(SEED_BASE - 2)
    Ws, bs, gs = init_rms(B_GROUP, g, dev)
    params = Ws + bs + gs
    for p in params:
        p.requires_grad_(True)
    targets = torch.randint(0, 2, (B_GROUP, N_TARGETS, D), generator=g, dtype=torch.float32, device=dev)
    shifts = torch.arange(D, device=dev, dtype=torch.int64); code_t = (targets.to(torch.int64) << shifts).sum(-1)
    pw = torch.tensor(5.0, device=dev); opt = torch.optim.Adam(params, lr=LR_START)
    torch.cuda.synchronize(dev); t0 = None
    for i in range(k + 5):
        if i == 5:
            torch.cuda.synchronize(dev); t0 = time.time()
        x, y = sample_natural(targets, BATCH, RHO, P_BG, g, dev, shifts, code_t)
        loss = F.binary_cross_entropy_with_logits(fwd_batched(Ws, bs, gs, x), y, pos_weight=pw)
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP); opt.step()
    torch.cuda.synchronize(dev)
    return (time.time() - t0) / k

_counter = {"g": 0}; _lock = threading.Lock()
_stats = {"made": 0, "fail": 0, "minpos_sum": 0.0}

def next_gid():
    with _lock:
        gid = _counter["g"]; _counter["g"] += 1
        return gid

def worker(rank, dev, t_start, budget_s):
    while time.time() - t_start < budget_s:
        gid = next_gid(); seed = SEED_BASE + gid; tg = time.time()
        Ws, bs, gs, targets, min_pos = train_group(B_GROUP, STEPS, seed, dev)
        save_shard(gid, Ws, bs, gs, targets, min_pos, seed)
        nfail = int((min_pos < 0.99).sum().item())
        with _lock:
            _stats["made"] += B_GROUP; _stats["fail"] += nfail
            _stats["minpos_sum"] += float(min_pos.sum().item())
        el = (time.time() - t_start) / 3600
        print(f"[gpu{rank}] gid {gid:05d} | {B_GROUP} in {time.time()-tg:.0f}s | "
              f"min_pos {min_pos.mean():.4f}/{min_pos.min():.4f} | <0.99:{nfail} | "
              f"total {_stats['made']} | {el:.2f}h", flush=True)
        del Ws, bs, gs, targets, min_pos
        torch.cuda.empty_cache()

ngpu = torch.cuda.device_count()
print(f"GPUs {ngpu} | out {OUT_DIR} | D={D} H={H} N={N_TARGETS} steps={STEPS} B_GROUP={B_GROUP}", flush=True)
dt = calibrate("cuda:0"); group_s = dt * STEPS; budget_s = BUDGET_HOURS * 3600
proj = int(budget_s / group_s) * B_GROUP * ngpu
print(f"calibration: {dt*1000:.1f} ms/step -> {group_s/60:.1f} min/group of {B_GROUP}", flush=True)
print(f"projection: ~{proj} organisms in {BUDGET_HOURS}h on {ngpu} GPU(s)\n", flush=True)

t_start = time.time()
threads = [threading.Thread(target=worker, args=(r, f"cuda:{r}", t_start, budget_s)) for r in range(ngpu)]
for t in threads: t.start()
for t in threads: t.join()

shards = sorted(f for f in os.listdir(OUT_DIR) if f.startswith("shard_") and f.endswith(".pt"))
manifest = {"n_shards": len(shards), "n_organisms": _stats["made"], "n_failed_memorize": _stats["fail"],
            "mean_min_pos": _stats["minpos_sum"] / max(1, _stats["made"]), "shards": shards,
            "config": {"D": D, "H": H, "N": N_TARGETS, "steps": STEPS, "B_GROUP": B_GROUP,
                       "BATCH": BATCH, "P_BG": P_BG, "RHO": RHO, "act": "relu2"}}
with open(os.path.join(OUT_DIR, "manifest.json"), "w") as fh:
    json.dump(manifest, fh, indent=2)
print(f"\nDONE: {_stats['made']} organisms in {len(shards)} shards | "
      f"failed-memorize {_stats['fail']} ({100*_stats['fail']/max(1,_stats['made']):.2f}%) | "
      f"mean min_pos {manifest['mean_min_pos']:.4f}")

GPUs 2 | out /kaggle/working/dataset64 | D=64 H=128 N=16 steps=40000 B_GROUP=256
calibration: 54.8 ms/step -> 36.6 min/group of 256
projection: ~7168 organisms in 8.6h on 2 GPU(s)

[gpu1] gid 00001 | 256 in 2160s | min_pos 1.0000/0.9979 | <0.99:0 | total 256 | 0.60h
[gpu0] gid 00000 | 256 in 2190s | min_pos 0.9999/0.9903 | <0.99:0 | total 512 | 0.61h
[gpu1] gid 00002 | 256 in 2156s | min_pos 1.0000/0.9966 | <0.99:0 | total 768 | 1.20h
[gpu0] gid 00003 | 256 in 2189s | min_pos 1.0000/0.9993 | <0.99:0 | total 1024 | 1.22h
[gpu1] gid 00004 | 256 in 2162s | min_pos 0.9999/0.9683 | <0.99:1 | total 1280 | 1.80h
[gpu0] gid 00005 | 256 in 2189s | min_pos 1.0000/0.9961 | <0.99:0 | total 1536 | 1.82h
[gpu1] gid 00006 | 256 in 2162s | min_pos 1.0000/0.9991 | <0.99:0 | total 1792 | 2.40h
[gpu0] gid 00007 | 256 in 2190s | min_pos 0.9999/0.9873 | <0.99:1 | total 2048 | 2.43h
[gpu1] gid 00008 | 256 in 2156s | min_pos 1.0000/0.9974 | <0.99:0 | total 2304 | 3.00h
[gpu0] gid 00009 | 256 in 2190s | min_p